In [4]:
"""
Daniusis Benchmark — Flow Matching RCA (v3)
Last Modified: June 10, 2026

Fixes from v2:
  1. Flow direction: faulty → normal (t=0 is faulty, t=1 is normal)
  2. Velocity extracted at t=0 (faulty state) consistently
  3. SIGMA=0.2 in single mode (consistent with benchmark)
  4. Reports both H and R side by side
  5. E[v_Z] diagnostic consistent with flow direction

Score: R(Z) = H(v_Z) - log(sigma_Z)
       H(Z) = H(v_Z)
Root cause = argmax R(Z) or argmax H(Z)
"""

import torch
import torch.nn as nn
import numpy as np
from scipy.stats import norm, gaussian_kde

device = "cuda" if torch.cuda.is_available() else "cpu"

# ==========================================
# 1. INPUT DISTRIBUTIONS
# ==========================================
def sample_pX(dist, n, sigma=0.2):
    if dist == "A":
        x = np.random.uniform(0, 1, n)
    elif dist == "B":
        x = np.random.normal(0, sigma, n)
    elif dist == "C":
        x = np.random.normal(0.5, sigma, n)
    elif dist == "D":
        x = np.random.normal(1.0, sigma, n)
    elif dist == "E":
        which = np.random.binomial(1, 0.5, n)
        x     = which * np.random.normal(0.3, sigma, n) + \
                (1 - which) * np.random.normal(0.7, sigma, n)
    return np.clip(x, 0, 1)

# ==========================================
# 2. MECHANISMS
# ==========================================
def make_s5():
    alphas = np.random.dirichlet(np.ones(5))
    mus    = np.random.uniform(0, 1, 5)
    sigmas = np.random.uniform(0, 0.1, 5)

    def s5(x):
        result = np.zeros_like(x)
        for i in range(5):
            result += alphas[i] * norm.cdf(x, loc=mus[i],
                                           scale=sigmas[i] + 1e-6)
        return result
    return s5

def apply_f(x, func, s5_fn=None):
    if func == "a":
        return np.cbrt(x)
    elif func == "b":
        return np.sqrt(np.abs(x) + 1e-8)
    elif func == "c":
        return x**2
    elif func == "d":
        return x**3
    elif func == "e":
        return s5_fn(x)

# ==========================================
# 3. DATA GENERATION
# ==========================================
def generate_pair(n, px_dist, func, s5_fn,
                  fault_magnitude=1.0, lam=0.1, sigma=0.2):
    def make_data(fault_x):
        x = sample_pX(px_dist, n, sigma=sigma) + fault_x
        x = np.clip(x, 0, 1 + fault_x)
        y = apply_f(x, func, s5_fn) + \
            lam * np.random.normal(0, sigma, n)
        X = torch.tensor(x, dtype=torch.float32).unsqueeze(1).to(device)
        Y = torch.tensor(y, dtype=torch.float32).unsqueeze(1).to(device)
        return torch.cat([X, Y], dim=1)

    # Returns (normal, faulty)
    return make_data(0.0), make_data(fault_magnitude)

# ==========================================
# 4. FLOW MODEL
# ==========================================
class FlowModel2D(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(3, 128), nn.ELU(),
            nn.Linear(128, 128), nn.ELU(),
            nn.Linear(128, 128), nn.ELU(),
            nn.Linear(128, 2)
        )

    def forward(self, x, t):
        return self.net(torch.cat([x, t], dim=1))

# ==========================================
# 5. TRAINING — faulty → normal
# ==========================================
def train_flow(p_n_fn, p_f_fn, epochs=1000, batch=256):
    """
    Flow direction: faulty → normal
      t=0: faulty state
      t=1: normal state
      x_t = (1-t)*x_faulty + t*x_normal
      v_target = x_normal - x_faulty
    """
    model = FlowModel2D().to(device)
    opt   = torch.optim.Adam(model.parameters(), lr=1e-3)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

    for epoch in range(epochs):
        p_n      = p_n_fn(batch)
        p_f      = p_f_fn(batch)
        t        = torch.rand(batch, 1).to(device)
        # Flow: faulty → normal
        x_t      = (1 - t) * p_f + t * p_n
        v_target = p_n - p_f
        loss     = nn.functional.mse_loss(model(x_t, t), v_target)
        opt.zero_grad(); loss.backward(); opt.step(); sched.step()

        if epoch % 200 == 0:
            print(f"  Epoch {epoch:>4} | Loss: {loss.item():.4f}")

    return model

# ==========================================
# 6. AUDIT — velocity at t=0 (faulty state)
# ==========================================
def compute_entropy(v):
    kde = gaussian_kde(v)
    return -np.mean(kde.logpdf(v))


def audit(model, p_n, p_f, node_names=["X", "Y"], verbose=False):
    """
    Extract velocity at t=0 (faulty state).
    Compute both H(v_Z) and R(Z) = H(v_Z) - log(sigma_Z).
    """
    model.eval()
    n = len(p_f)
    with torch.no_grad():
        # Evaluate at t=0 (faulty state)
        t    = torch.ones(n, 1).to(device)
        t = t * 0
        vels = model(p_f, t).cpu().numpy()

    # Compute scores
    h_scores = {}
    r_scores = {}
    entropies = {}
    sigmas = {}
    e_v = {}

    for i, name in enumerate(node_names):
        v_i = vels[:, i]
        entropies[name] = compute_entropy(v_i)
        sigmas[name]    = np.std(v_i)
        e_v[name]       = np.mean(v_i)
        h_scores[name]  = entropies[name]

        if sigmas[name] < 1e-8:
            r_scores[name] = -np.inf
        else:
            r_scores[name] = entropies[name] - np.log(sigmas[name])

    pred_h = max(h_scores, key=h_scores.get)
    pred_r = max(r_scores, key=r_scores.get)

    if verbose:
        print(f"\n  {'Node':<6} {'H(v)':>10} {'sigma':>10} "
              f"{'log(sigma)':>12} {'R(Z)':>10} {'E[v]':>10}")
        print(f"  {'-'*62}")
        for name in node_names:
            if sigmas[name] < 1e-8:
                print(f"  {name:<6} {entropies[name]:>10.4f} "
                      f"{'—':>10} {'—':>12} {'degen':>10} "
                      f"{e_v[name]:>10.4f}")
            else:
                print(f"  {name:<6} {entropies[name]:>10.4f} "
                      f"{sigmas[name]:>10.4f} "
                      f"{np.log(sigmas[name]):>12.4f} "
                      f"{r_scores[name]:>10.4f} "
                      f"{e_v[name]:>10.4f}")

        print(f"\n  PREDICTED (H): {pred_h}")
        print(f"  PREDICTED (R): {pred_r}")
        print(f"\n  E[v_X] = {e_v['X']:.4f}  "
              f"({'≈0 ✓' if abs(e_v['X']) < 0.1 else '≠0 ✗'})")
        print(f"  E[v_Y] = {e_v['Y']:.4f}  "
              f"({'≈0 ✓' if abs(e_v['Y']) < 0.1 else '≠0 ✗'})")

    return pred_h, pred_r, h_scores, r_scores

# ==========================================
# 7. SINGLE EXPERIMENT
# ==========================================
def run_experiment(px_dist, func, n=1000,
                   fault_magnitude=1.0, lam=0.1, sigma=0.2,
                   epochs=1000, verbose=False):
    s5_fn = make_s5() if func == "e" else None

    def p_n_fn(batch):
        p_n, _ = generate_pair(batch, px_dist, func, s5_fn,
                               fault_magnitude, lam, sigma)
        return p_n

    def p_f_fn(batch):
        _, p_f = generate_pair(batch, px_dist, func, s5_fn,
                               fault_magnitude, lam, sigma)
        return p_f

    model = train_flow(p_n_fn, p_f_fn, epochs=epochs)
    p_n, p_f = generate_pair(n, px_dist, func, s5_fn,
                             fault_magnitude, lam, sigma)

    pred_h, pred_r, h_scores, r_scores = audit(
        model, p_n, p_f, verbose=verbose
    )

    correct_h = pred_h == "X"
    correct_r = pred_r == "X"

    return correct_h, correct_r, pred_h, pred_r, h_scores, r_scores

# ==========================================
# 8. BENCHMARK
# ==========================================
PX_DISTS = ["A", "B", "C", "D", "E"]
FUNCS    = ["a", "b", "c", "d", "e"]

PX_NAMES = {
    "A": "U(0,1)",
    "B": "N(0,σ²)",
    "C": "N(0.5,σ²)",
    "D": "N(1,σ²)",
    "E": "GM([0.3,0.7])"
}

FUNC_NAMES = {
    "a": "x^(1/3)",
    "b": "x^(1/2)",
    "c": "x²",
    "d": "x³",
    "e": "s5(x)"
}

def run_benchmark(n=1000, fault_magnitude=1.0, lam=0.1,
                  sigma=0.2, epochs=1000, repeats=100):
    print("\n" + "█" * 72)
    print("  DANIUSIS ET AL. BENCHMARK — FLOW MATCHING RCA (v3)")
    print(f"  Flow: faulty → normal | Velocity at t=0")
    print(f"  n={n}, fault={fault_magnitude}, λ={lam}, "
          f"σ={sigma}, epochs={epochs}, repeats={repeats}")
    print("█" * 72)

    print(f"\n  {'p_X':<18} {'f(x)':<12} "
          f"{'H Acc':>8}  {'R Acc':>8}")
    print(f"  {'-'*50}")

    all_h_results = {}
    all_r_results = {}

    for px in PX_DISTS:
        for f in FUNCS:
            h_correct_count = 0
            r_correct_count = 0
            for _ in range(repeats):
                correct_h, correct_r, _, _, _, _ = run_experiment(
                    px_dist=px, func=f, n=n,
                    fault_magnitude=fault_magnitude,
                    lam=lam, sigma=sigma, epochs=epochs
                )
                if correct_h:
                    h_correct_count += 1
                if correct_r:
                    r_correct_count += 1

            h_acc = h_correct_count / repeats * 100
            r_acc = r_correct_count / repeats * 100
            all_h_results[(px, f)] = h_acc
            all_r_results[(px, f)] = r_acc
            print(f"  {PX_NAMES[px]:<18} {FUNC_NAMES[f]:<12} "
                  f"{h_acc:>7.1f}%  {r_acc:>7.1f}%")

    overall_h = np.mean(list(all_h_results.values()))
    overall_r = np.mean(list(all_r_results.values()))
    print(f"\n  {'─' * 50}")
    print(f"  {'Overall H Accuracy':<30} {overall_h:>7.1f}%")
    print(f"  {'Overall R Accuracy':<30} {overall_r:>7.1f}%")

    # Summary tables
    for label, results in [("H", all_h_results), ("R", all_r_results)]:
        print(f"\n  {label} ACCURACY TABLE (rows=p_X, cols=f)")
        print(f"  {'':18}", end="")
        for f in FUNCS:
            print(f"  {FUNC_NAMES[f]:>9}", end="")
        print()
        print(f"  {'-' * 70}")
        for px in PX_DISTS:
            print(f"  {PX_NAMES[px]:<18}", end="")
            for f in FUNCS:
                print(f"  {results[(px, f)]:>8.1f}%", end="")
            print()

    print("█" * 72)
    return all_h_results, all_r_results

# ==========================================
# 9. MAIN — Jupyter compatible
# ==========================================

# ── Configure here ────────────────────────
MODE           = "single"    # "single" or "benchmark"

# Single experiment settings
PX             = "A"         # A, B, C, D, E
FUNC           = "a"         # a, b, c, d, e
FAULT          = 1.0
LAM            = 0.1
SIGMA          = 0.2         # Fixed: was 2, now 0.2
EPOCHS         = 1000
N              = 1000

# Benchmark additional settings
REPEATS        = 100         # repeats per combination
# ─────────────────────────────────────────

if MODE == "single":
    print(f"\n  Single experiment")
    print(f"  p_X={PX_NAMES[PX]}, f={FUNC_NAMES[FUNC]}, "
          f"fault={FAULT}, λ={LAM}, σ={SIGMA}, epochs={EPOCHS}")
    print(f"  Flow: faulty → normal | Velocity at t=0")

    correct_h, correct_r, pred_h, pred_r, h_scores, r_scores = \
        run_experiment(
            px_dist=PX, func=FUNC, n=N,
            fault_magnitude=FAULT, lam=LAM,
            sigma=SIGMA, epochs=EPOCHS, verbose=True
        )

    print(f"\n  True RC  : X")
    print(f"  Pred (H) : {pred_h}  "
          f"{'✓' if correct_h else '✗'}")
    print(f"  Pred (R) : {pred_r}  "
          f"{'✓' if correct_r else '✗'}")

elif MODE == "benchmark":
    all_h, all_r = run_benchmark(
        n=N, fault_magnitude=FAULT, lam=LAM,
        sigma=SIGMA, epochs=EPOCHS, repeats=REPEATS
    )


  Single experiment
  p_X=U(0,1), f=x^(1/3), fault=1.0, λ=0.1, σ=0.2, epochs=1000
  Flow: faulty → normal | Velocity at t=0
  Epoch    0 | Loss: 0.7605
  Epoch  200 | Loss: 0.0455
  Epoch  400 | Loss: 0.0456
  Epoch  600 | Loss: 0.0453
  Epoch  800 | Loss: 0.0363

  Node         H(v)      sigma   log(sigma)       R(Z)       E[v]
  --------------------------------------------------------------
  X          0.3520     0.3625      -1.0146     1.3666    -1.0020
  Y         -0.7534     0.1154      -2.1597     1.4063    -0.3782

  PREDICTED (H): X
  PREDICTED (R): Y

  E[v_X] = -1.0020  (≠0 ✗)
  E[v_Y] = -0.3782  (≠0 ✗)

  True RC  : X
  Pred (H) : X  ✓
  Pred (R) : Y  ✗
